In [ ]:
import os
import time
import subprocess
import pandas as pd
from pyspark.sql import SparkSession


In [ ]:
#HDFS paths

stream_input_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/input/"
temp_batches_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/temp_batches/"

#This file is splitted into batches and sent to the stream input directory
source_file = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/dataset/LI-Medium_Trans.csv"

In [ ]:
#Create HDFS directories for streaming input and checkpointing
#subprocess.run(["hdfs", "dfs", "-mkdir", "-p", hdfs_input_path], check=True)
#subprocess.run(["hdfs", "dfs", "-mkdir", "-p", hdfs_checkpoint_path], check=True)

In [ ]:
#Clean input directory
#subprocess.run(["hdfs", "dfs", "-rm", "-f", f"{hdfs_input_path}/*.csv"], check=False)

In [ ]:
#Initialize Spark session
spark = SparkSession.builder \
    .appName("AML Streaming Producer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(source_file)

print("Total records:", df.count())
df.show(5, truncate=False)

In [ ]:
#Create x batches from the source file
num_batches = 5

batches = df.randomSplit([1.0] * num_batches, seed=42)

print(f"Criados {num_batches} batches.")

In [ ]:
#Send batches to stream input directory with a delay between them

delay_seconds = 10

for i, batch_df in enumerate(batches, start=1):
    batch_path = f"{stream_input_path}/batch_{i:03d}"

    print(f"Enviando batch {i} para {batch_path}...")

    batch_df.coalesce(1) \
        .write \
        .mode("append") \
        .option("header", True) \
        .csv(batch_path)

    print(f"Batch {i} enviado.")
    time.sleep(delay_seconds)

print("Producer terminou.")